In [ ]:
# import os
# import json
# import pandas as pd
# import numpy as np
# import xgboost as xgb
# from datetime import datetime, timedelta
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import StandardScaler
# from sklearn.pipeline import Pipeline
# from scipy.stats import uniform, randint
# from sklearn.metrics import mean_squared_error
# import cloudpickle
# import io
# import platform
# import sys

# import warnings
# warnings.filterwarnings("ignore")

# # GCP/Cloud deployment compatibility settings
# def setup_cloud_compatibility():
#     """Setup settings for cloud deployment compatibility"""
#     # Disable GPU for XGBoost to avoid CUDA issues on GCP
#     os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
    
#     # Set threading for cloud environments
#     os.environ['OMP_NUM_THREADS'] = '2'
#     os.environ['MKL_NUM_THREADS'] = '2'
    
#     # Print environment info for debugging
#     print(f"Python version: {sys.version}")
#     print(f"Platform: {platform.platform()}")
#     print(f"XGBoost version: {xgb.__version__}")
#     print(f"GPU disabled for cloud compatibility")

# # Setup cloud compatibility
# setup_cloud_compatibility()

# # Define functions for M2
# def calculate_current_month(row):
#     cohort_start = row['cohort']
#     elapsed_months = row['elapsed_months']
#     return (cohort_start + pd.DateOffset(months=int(elapsed_months))).month

# def quantile_loss(alpha):
#     def custom_loss(y_true, y_pred):
#         errors = y_pred - y_true
#         grad = np.where(errors < 0, -alpha, 1 - alpha)
#         hess = np.ones_like(y_true)
#         return grad, hess
#     return custom_loss

# def model_train_save_pipeline(game, alpha, X_train, y_train, X_test=None, y_test=None):
#     # Cloud-optimized XGBoost settings
#     model = xgb.XGBRegressor(
#         objective=quantile_loss(alpha), 
#         enable_categorical=True, 
#         random_state=42,
#         tree_method='hist',  # More memory efficient for cloud
#         n_jobs=2,  # Limit parallel jobs for cloud stability
#         gpu_id=-1  # Explicitly disable GPU
#     )  
#     numerical_columns = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
    
#     # Convert categorical columns to numeric for consistency (minimal fix)
#     X_processed = X_train.copy()
#     categorical_columns = X_train.select_dtypes(include=['category']).columns.tolist()
#     for col in categorical_columns:
#         X_processed[col] = X_processed[col].cat.codes
    
#     preprocessor = ColumnTransformer(
#         transformers=[('num', StandardScaler(), numerical_columns)],
#         remainder='passthrough'
#     )
#     pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    
#     # Exact same hyperparameters as playground for consistency
#     param_dist = {
#         'model__n_estimators': randint(50, 300),  # Same as playground
#         'model__learning_rate': uniform(0.01, 0.09),
#         'model__max_depth': randint(3, 8),  # Same as playground
#         'model__subsample': uniform(0.6, 0.4),
#         'model__colsample_bytree': uniform(0.6, 0.4),
#         'model__min_child_weight': randint(1, 6),  # Same as playground
#         'model__gamma': uniform(0, 0.2),  # Same as playground
#         'model__reg_alpha': uniform(0, 0.5),  # Same as playground
#         'model__reg_lambda': uniform(1, 1.5),  # Same as playground
#     }
    
#     # Same RandomizedSearchCV settings as playground (with cloud fallback)
#     random_search = RandomizedSearchCV(
#         pipeline, 
#         param_distributions=param_dist,
#         n_iter=20,  # Same as playground
#         scoring='neg_mean_squared_error',
#         cv=3,  # Same as playground
#         n_jobs=-1,  # Same as playground (cloud will handle gracefully)
#         random_state=42,
#         verbose=0  # Reduce output for cloud logs
#     )
    
#     try:
#         random_search.fit(X_processed, y_train)
#         best_model = random_search.best_estimator_
#     except Exception as e:
#         print(f"RandomizedSearchCV failed for {game}, falling back to default model: {e}")
#         # Fallback to default model if hyperparameter search fails
#         pipeline.fit(X_processed, y_train)
#         best_model = pipeline
    
#     # Save model with error handling
#     try:
#         with open(f'{game}_xgb_{alpha}.pkl', 'wb') as f:
#             cloudpickle.dump(best_model, f)
#         print(f"Model saved successfully for {game}_alpha_{alpha}")
#     except Exception as e:
#         print(f"Warning: Could not save model for {game}_alpha_{alpha}: {e}")
    
#     # Safe feature names storage
#     try:
#         best_model.__dict__["feature_names_in_"] = X_processed.columns.tolist()
#     except Exception as e:
#         print(f"Warning: Could not store feature names for {game}: {e}")
    
#     # Calculate metrics with error handling
#     try:
#         y_train_pred = best_model.predict(X_processed)
#         rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
#     except Exception as e:
#         print(f"Warning: Could not calculate training RMSE for {game}: {e}")
#         rmse_train = float('inf')
    
#     result = {"model": best_model, "rmse_train": rmse_train}
    
#     if X_test is not None and y_test is not None:
#         try:
#             # Process X_test the same way as X_train
#             X_test_processed = X_test.copy()
#             for col in categorical_columns:
#                 if col in X_test_processed.columns:
#                     X_test_processed[col] = X_test_processed[col].cat.codes
#             y_test_pred = best_model.predict(X_test_processed)
#             rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
#             result["rmse_test"] = rmse_test
#         except Exception as e:
#             print(f"Warning: Could not calculate test RMSE for {game}: {e}")
#             result["rmse_test"] = float('inf')
    
#     return result

# def augment_data(X, y, augmentation_factor, variation_pct, random_seed=42):
#     np.random.seed(random_seed)
#     n_aug = int((augmentation_factor - 1)*len(X))
#     if n_aug <= 0:
#         return X, y
    
#     # Memory optimization for cloud
#     if n_aug > 10000:  # Limit augmentation for cloud memory
#         print(f"Warning: Limiting augmentation to 10000 samples for cloud compatibility")
#         n_aug = 10000
    
#     numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col]) and col not in ['cohort_month','current_month']]
#     X_aug_list, y_aug_list = [], []
    
#     for _ in range(n_aug):
#         idx = np.random.randint(0, len(X))
#         x_row = X.iloc[idx].copy()
#         for col in numeric_cols:
#             sign = np.random.choice([-1,1])
#             x_row[col] = x_row[col] * (1 + sign*(variation_pct/100.0))
#         X_aug_list.append(x_row)
#         y_val = y.iloc[idx]
#         sign = np.random.choice([-1,1])
#         y_aug_list.append(y_val * (1 + sign*(variation_pct/100.0)))
    
#     X_aug = pd.DataFrame(X_aug_list)
#     y_aug = pd.Series(y_aug_list, index=X_aug.index)
#     X_final = pd.concat([X, X_aug], ignore_index=True)
#     y_final = pd.concat([y, y_aug], ignore_index=True)
#     return X_final, y_final

# def preprocess_m2(game):
#     try:
#         df = pd.read_csv(f'Latest Data/Segregated/{game}_Dataset_12_2024.csv')
#     except FileNotFoundError:
#         # Try alternative path for cloud environments
#         try:
#             df = pd.read_csv(f'{game}_Dataset_12_2024.csv')
#         except FileNotFoundError:
#             raise FileNotFoundError(f"Could not find dataset for {game} in expected locations")
    
#     df['elapsed_months'].fillna(1, inplace=True)
#     df = df[(df['total_cac'] != 0) & (df['total_cac'].notna())]
#     df['elapsed_months'] -= 1
#     df = df[df['elapsed_months'] > 0]
#     df['total_cac'] = df.apply(lambda row: row['client_actual_spend'] if (pd.isnull(row['total_cac']) or row['total_cac']==0) and not pd.isnull(row['client_actual_spend']) else row['total_cac'], axis=1)
#     df['client_actual_spend'] = df.apply(lambda row: row['total_cac'] if (pd.isnull(row['client_actual_spend']) or row['client_actual_spend']==0) and not pd.isnull(row['total_cac']) else row['client_actual_spend'], axis=1)
#     cl1 = df.loc[:, df.columns.str.startswith('MC_')].columns.tolist()
#     df = df[~df['cohort'].str.endswith(('2021','2022','2020','2019','2018'))]
#     df = df[~df['cohort'].str.startswith(('2021','2022','2020','2019','2018'))]
#     if game == "G090":
#         df['cohort'] = df['cohort'].str.replace('-', '/')
#         df['transaction_period'] = df['transaction_period'].str.replace('-', '/')
#     df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
#     df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce')
#     df['cohort_month'] = df['cohort'].dt.month
#     df['current_month'] = df.apply(calculate_current_month, axis=1)
#     df['cohort_month'] = df['cohort_month'].astype('category')
#     df['current_month'] = df['current_month'].astype('category')
#     current_month_start = datetime.now().replace(day=1)
#     current_month_start = current_month_start.strftime('%d/%m/%Y').lstrip('0').replace('/0','/')
#     filtered_df = df[pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start].copy()
#     feature_columns = ['day_1_cumulative_ltv_cac', 'day_14_net_revenue_retention', 'day_7_cumulative_ltv_cac', 
#                        'day_14_cumulative_ltv_cac', 'day_30_cumulative_ltv_cac', 'client_actual_spend',
#                        'cohort_month','current_month','m0_ltv_cac'] + cl1
#     for lag in range(1,4):
#         df[f'ltv_lag_{lag}'] = df.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)
#     dft = df[pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start].copy()
#     return dft, feature_columns, df

# def forecast_cumulative_ltv_cac_variations_m2(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12, epsilon=0.001):
#     game_data = df[df['game_id'] == game_id].copy()
#     if game_id == "G090":
#         game_data = game_data[game_data['cohort'] >= pd.Timestamp("2024-12-01")]
#     forecasted_results = []
#     sp_ltv_cac = df[df['transaction_period_ltv_cac'] > 0]['transaction_period_ltv_cac'].min()
#     for cohort in game_data['cohort'].unique():
#         cohort_data = game_data[game_data['cohort'] == cohort].copy()
#         m0_ltv_cac = cohort_data['m0_ltv_cac'].iloc[0]
#         total_cac = cohort_data['client_actual_spend'].iloc[0]
#         last_elapsed_month = int(cohort_data['elapsed_months'].max())
#         cumulative_actual = m0_ltv_cac
#         actual_cumulative = []
#         for month in range(1, last_elapsed_month + 1):
#             historical_data = cohort_data.loc[cohort_data['elapsed_months'] == month, 'transaction_period_ltv_cac']
#             transactional_actual = historical_data.values[0] if not historical_data.empty else 0
#             cumulative_actual += transactional_actual
#             actual_cumulative.append({
#                 'month': month,
#                 'transactional_actual': transactional_actual,
#                 'cumulative_actual': cumulative_actual,
#                 'absolute_revenue_actual': total_cac * transactional_actual,
#                 'total_cac': total_cac
#             })
#         lag_1 = np.nan; lag_2 = np.nan; lag_3 = np.nan
#         lag_1b = np.nan; lag_1g = np.nan; lag_2b = np.nan; lag_2g = np.nan; lag_3b = np.nan; lag_3g = np.nan
#         cumulative_values = {key: m0_ltv_cac for key in ['cumulative_bad', 'cumulative_expected', 'cumulative_good']}
        
#         # Safe feature names access for cloud compatibility
#         try:
#             feature_names = best_model_expected.__dict__["feature_names_in_"]
#         except (KeyError, AttributeError):
#             # Fallback for different XGBoost versions
#             feature_names = best_model_expected.feature_names_in_ if hasattr(best_model_expected, 'feature_names_in_') else None
#             if feature_names is None:
#                 print(f"Warning: Could not access feature names for {game_id}, using fallback")
#                 continue
        
#         static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month, feature_names].iloc[0].copy()
#         for col in ['elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']:
#             if col in static_features:
#                 static_features = static_features.drop(col)
#         for i in range(1, last_elapsed_month + forecast_months + 1):
#             elapsed_months = i
#             features = static_features.copy()
#             features['elapsed_months'] = elapsed_months
            
#             # Safe feature check
#             try:
#                 if 'current_month' in feature_names:
#                     cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
#                     current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
#                     features['current_month'] = current_date.month
#                 else:
#                     current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)
#             except Exception as e:
#                 print(f"Warning: Error setting current_month for {game_id}: {e}")
#                 current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)
            
#             if i <= last_elapsed_month:
#                 historical_data = cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac']
#                 if historical_data.empty:
#                     lag_value = epsilon
#                 else:
#                     historical_value = historical_data.values[0]
#                     lag_value = historical_value if historical_value != 0 else epsilon
#                 lag_1 = lag_value
#                 lag_1b = lag_value
#                 lag_1g = lag_value
#                 features1 = features.copy()
#                 features2 = features.copy()
#                 features['ltv_lag_1'] = lag_1
#                 features1['ltv_lag_1'] = lag_1b
#                 features2['ltv_lag_1'] = lag_1g
#                 features['ltv_lag_2'] = lag_2
#                 features1['ltv_lag_2'] = lag_2b
#                 features2['ltv_lag_2'] = lag_2g
#                 features['ltv_lag_3'] = lag_3
#                 features1['ltv_lag_3'] = lag_3b
#                 features2['ltv_lag_3'] = lag_3g
#                 features_df = pd.DataFrame([features])
#                 features_df1 = pd.DataFrame([features1])
#                 features_df2 = pd.DataFrame([features2])
                
#                 # Apply categorical conversion for prediction consistency
#                 for col in ['cohort_month', 'current_month']:
#                     if col in features_df.columns:
#                         if features_df[col].dtype == 'category':
#                             features_df[col] = features_df[col].cat.codes
#                         if features_df1[col].dtype == 'category':
#                             features_df1[col] = features_df1[col].cat.codes
#                         if features_df2[col].dtype == 'category':
#                             features_df2[col] = features_df2[col].cat.codes
                
#                 try:
#                     predicted_val_bad = max(0, best_model_bad.predict(features_df1)[0])
#                     predicted_val_expected = max(0, best_model_expected.predict(features_df)[0])
#                     predicted_val_good = max(0, best_model_good.predict(features_df2)[0])
#                 except Exception as e:
#                     print(f"Warning: Prediction error for {game_id} at month {i}: {e}")
#                     predicted_val_bad = predicted_val_expected = predicted_val_good = epsilon
                
#                 x = [predicted_val_bad, predicted_val_expected, predicted_val_good]
#                 x.sort()
#                 predicted_val_bad, predicted_val_expected, predicted_val_good = x
#                 if predicted_val_good <= 0:
#                     predicted_val_good = sp_ltv_cac
#                 if predicted_val_expected <= 0:
#                     predicted_val_expected = predicted_val_good / 2
#                 if predicted_val_bad <= 0:
#                     predicted_val_bad = predicted_val_expected / 2
#             else:
#                 features1 = features.copy()
#                 features2 = features.copy()
#                 features['ltv_lag_1'] = lag_1
#                 features1['ltv_lag_1'] = lag_1b
#                 features2['ltv_lag_1'] = lag_1g
#                 features['ltv_lag_2'] = lag_2
#                 features1['ltv_lag_2'] = lag_2b
#                 features2['ltv_lag_2'] = lag_2g
#                 features['ltv_lag_3'] = lag_3
#                 features1['ltv_lag_3'] = lag_3b
#                 features2['ltv_lag_3'] = lag_3g
#                 features_df = pd.DataFrame([features])
#                 features_df1 = pd.DataFrame([features1])
#                 features_df2 = pd.DataFrame([features2])
                
#                 # Apply categorical conversion for prediction consistency
#                 for col in ['cohort_month', 'current_month']:
#                     if col in features_df.columns:
#                         if features_df[col].dtype == 'category':
#                             features_df[col] = features_df[col].cat.codes
#                         if features_df1[col].dtype == 'category':
#                             features_df1[col] = features_df1[col].cat.codes
#                         if features_df2[col].dtype == 'category':
#                             features_df2[col] = features_df2[col].cat.codes
                
#                 try:
#                     predicted_val_bad = max(0, best_model_bad.predict(features_df1)[0])
#                     predicted_val_expected = max(0, best_model_expected.predict(features_df)[0])
#                     predicted_val_good = max(0, best_model_good.predict(features_df2)[0])
#                 except Exception as e:
#                     print(f"Warning: Prediction error for {game_id} at month {i}: {e}")
#                     predicted_val_bad = predicted_val_expected = predicted_val_good = epsilon
                
#                 x = [predicted_val_bad, predicted_val_expected, predicted_val_good]
#                 x.sort()
#                 predicted_val_bad, predicted_val_expected, predicted_val_good = x
#                 if predicted_val_good <= 0:
#                     predicted_val_good = sp_ltv_cac
#                 if predicted_val_expected <= 0:
#                     predicted_val_expected = predicted_val_good / 2
#                 if predicted_val_bad <= 0:
#                     predicted_val_bad = predicted_val_expected / 2
#                 lag_1b = predicted_val_bad
#                 lag_1 = predicted_val_expected
#                 lag_1g = predicted_val_good
#             lag_3 = lag_2
#             lag_2 = lag_1
#             lag_3b = lag_2b
#             lag_2b = lag_1b
#             lag_3g = lag_2g
#             lag_2g = lag_1g
#             row_result = {
#                 'game_id': game_id,
#                 'cohort': cohort,
#                 'elapsed_months': elapsed_months,
#                 'current_month': current_date.strftime('%Y-%m'),
#             }
#             actual_data = next((ac for ac in actual_cumulative if ac['month'] == elapsed_months), None)
#             row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
#             row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
#             row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
#             row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac
#             for model_type, predicted_val in zip(['bad', 'expected', 'good'], [predicted_val_bad, predicted_val_expected, predicted_val_good]):
#                 cumulative_key = f"cumulative_{model_type}"
#                 cumulative_values[cumulative_key] += predicted_val
#                 row_result[f"transactional_{model_type}"] = predicted_val
#                 row_result[cumulative_key] = cumulative_values[cumulative_key]
#                 row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val
#             forecasted_results.append(row_result)
#     forecast_df = pd.DataFrame(forecasted_results)
#     return forecast_df

# # --- Main Execution for M2 ---
# if __name__ == "__main__":
#     np.random.seed(42)
    
#     # Try multiple config paths for cloud compatibility
#     config_paths = [
#         "Latest Data/Segregated/config.json",
#         "config.json",
#         "/app/config.json"  # Common cloud path
#     ]
    
#     global_config = {}
#     for config_path in config_paths:
#         try:
#             with open(config_path, "r") as f:
#                 global_config = json.load(f)
#             print(f"Config loaded from: {config_path}")
#             break
#         except FileNotFoundError:
#             continue
    
#     if not global_config:
#         print("No config found in any expected location.")
#         sys.exit(1)
    
#     # Lists to track failures and feature changes
#     failed_games = []
#     changed_features = {}

#     for key, cfg in global_config.items():
#         if key.startswith("M2_") and cfg.get("flag", 0)==1:
#             game_id = cfg["game_id"]
#             selected_features = cfg["selected_features"]
#             use_augmentation = cfg["use_augmentation"]
#             augmentation_factor = cfg.get("augmentation_factor", 1.0)
#             augmentation_pct = cfg.get("augmentation_pct", 0)
#             print(f"Processing M2 configuration for {game_id} ...")
#             try:
#                 dft, feature_columns, df_orig = preprocess_m2(game_id)
#                 missing_feats = [feat for feat in selected_features if feat not in dft.columns]
#                 if missing_feats:
#                     print(f"Warning: For game {game_id}, the following features are missing and will be ignored: {missing_feats}")
#                     changed_features[game_id] = missing_feats
#                     selected_features = [feat for feat in selected_features if feat in dft.columns]
#                 X = dft[selected_features + ['ltv_lag_1','ltv_lag_2','ltv_lag_3','elapsed_months']]
#                 y = dft['transaction_period_ltv_cac']
#                 X_train = X
#                 y_train = y
#                 if use_augmentation and augmentation_factor > 1:
#                     X_train, y_train = augment_data(X_train, y_train, augmentation_factor, augmentation_pct)
#                 alphas = [0.25, 0.5, 0.75]
#                 models_info = {}
#                 for alpha in alphas:
#                     print(f"Training model with alpha={alpha} for game {game_id} ...")
#                     result = model_train_save_pipeline(game_id, alpha, X_train, y_train)
#                     models_info[alpha] = result
#                     print(f"Model {alpha} - RMSE: {result['rmse_train']:.4f}")
#                 best_model_expected = models_info[0.5]["model"]
#                 best_model_bad = models_info[0.75]["model"]
#                 best_model_good = models_info[0.25]["model"]
                
#                 print(f"Generating forecasts for {game_id}...")
#                 forecast_df = forecast_cumulative_ltv_cac_variations_m2(game_id, best_model_expected, best_model_bad, best_model_good, df_orig, forecast_months=12)
#                 output_csv = f"{game_id}_predictions.csv"
#                 forecast_df.to_csv(output_csv, index=False)
#                 print(f"Predictions saved to {output_csv}")
#             except Exception as e:
#                 print(f"Error processing game {game_id}: {e}")
#                 import traceback
#                 traceback.print_exc()
#                 failed_games.append(game_id)
#                 continue

#     # Summary
#     print(f"\n{'='*50}")
#     print("TRAINING SUMMARY")
#     print(f"{'='*50}")
#     print(f"Environment: {platform.platform()}")
#     print(f"Successful games: {len(global_config) - len(failed_games)}")
    
#     if failed_games:
#         print("\nThe following games failed during training:")
#         for game in failed_games:
#             print(f"  - {game}")
    
#     if changed_features:
#         print("\nThe following games had missing features (these features were ignored and need to be revisited):")
#         for game, features in changed_features.items():
#             print(f"  - Game {game}: Missing features {features}")
    
#     print(f"\n{'='*50}") 

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
import os
import json
import pandas as pd
import numpy as np
import xgboost as xgb
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error
import cloudpickle
import io

import warnings
warnings.filterwarnings("ignore")

# Define functions for M2 similar to M1 but with the differences you require
def calculate_current_month(row):
    cohort_start = row['cohort']
    elapsed_months = row['elapsed_months']
    return (cohort_start + pd.DateOffset(months=int(elapsed_months))).month

def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)
        hess = np.ones_like(y_true)
        return grad, hess
    return custom_loss

def model_train_save_pipeline(game, alpha, X_train, y_train):
    model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True)
    numerical_columns = X_train.select_dtypes(include=['int64','float64']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('num', StandardScaler(), numerical_columns)],
        remainder='passthrough'
    )
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    param_dist = {
        'model__n_estimators': randint(50, 300),
        'model__learning_rate': uniform(0.01, 0.09),
        'model__max_depth': randint(3, 8),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4),
        'model__min_child_weight': randint(1, 6),
        'model__gamma': uniform(0, 0.2),
        'model__reg_alpha': uniform(0, 0.5),
        'model__reg_lambda': uniform(1, 1.5),
    }
    random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist,
                                         n_iter=20, scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
                                         cv=TimeSeriesSplit(n_splits=3), n_jobs=-1, random_state=42)
    random_search.fit(X_train, y_train)
    best_model = random_search.best_estimator_
    with open(f'{game}_xgb_{alpha}.pkl', 'wb') as f:
        cloudpickle.dump(best_model, f)
    best_model.__dict__["feature_names_in_"] = X_train.columns
    y_train_pred = best_model.predict(X_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    return {"model": best_model, "rmse_train": rmse_train}

def augment_data(X, y, augmentation_factor, variation_pct):
    n_aug = int((augmentation_factor - 1)*len(X))
    if n_aug <= 0:
        return X, y
    numeric_cols = [col for col in X.columns if pd.api.types.is_numeric_dtype(X[col]) and col not in ['cohort_month','current_month']]
    X_aug_list, y_aug_list = [], []
    for _ in range(n_aug):
        idx = np.random.randint(0, len(X))
        x_row = X.iloc[idx].copy()
        for col in numeric_cols:
            sign = np.random.choice([-1,1])
            x_row[col] = x_row[col] * (1 + sign*(variation_pct/100.0))
        X_aug_list.append(x_row)
        y_val = y.iloc[idx]
        sign = np.random.choice([-1,1])
        y_aug_list.append(y_val * (1 + sign*(variation_pct/100.0)))
    X_aug = pd.DataFrame(X_aug_list)
    y_aug = pd.Series(y_aug_list, index=X_aug.index)
    X_final = pd.concat([X, X_aug], ignore_index=True)
    y_final = pd.concat([y, y_aug], ignore_index=True)
    return X_final, y_final

def preprocess_m2(game):
    df = pd.read_csv(f'Latest Data/Segregated/{game}_Dataset_12_2024.csv')
    df['elapsed_months'].fillna(1, inplace=True)
    df = df[(df['total_cac'] != 0) & (df['total_cac'].notna())]
    df['elapsed_months'] -= 1
    df['total_cac'] = df.apply(lambda row: row['client_actual_spend'] if (pd.isnull(row['total_cac']) or row['total_cac']==0) and not pd.isnull(row['client_actual_spend']) else row['total_cac'], axis=1)
    df['client_actual_spend'] = df.apply(lambda row: row['total_cac'] if (pd.isnull(row['client_actual_spend']) or row['client_actual_spend']==0) and not pd.isnull(row['total_cac']) else row['client_actual_spend'], axis=1)
    df = df[~df['cohort'].str.endswith(('2021','2022','2020','2019','2018'))]
    df = df[~df['cohort'].str.startswith(('2021','2022','2020','2019','2018'))]
    # if game == "G090":
    #     df['cohort'] = df['cohort'].str.replace('-', '/')
    #     df['transaction_period'] = df['transaction_period'].str.replace('-', '/')
    df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    df['transaction_period'] = pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce')
    df['cohort_month'] = df['cohort'].dt.month
    df['current_month'] = df.apply(calculate_current_month, axis=1)
    df['cohort_month'] = df['cohort_month'].astype('category')
    df['current_month'] = df['current_month'].astype('category')
    # Sort by cohort and elapsed_months
    df = df.sort_values(by=['cohort', 'elapsed_months'])
    # Create lag features
    for lag in range(1,4):
        df[f'ltv_lag_{lag}'] = df.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)
    current_month_start = pd.Timestamp.today().normalize().replace(day=1)  # first day of current month as a Timestamp (fix C1)
    # Create training subset
    dft = df[(pd.to_datetime(df['transaction_period'], format='%d/%m/%Y', errors='coerce') < current_month_start) & (df['elapsed_months'] > 0)].copy()
    return dft, df

def forecast_cumulative_ltv_cac_variations_m2(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12, epsilon=0.001):
    game_data = df[df['game_id'] == game_id].copy()
    if game_id == "G090":
        game_data = game_data[game_data['cohort'] >= pd.Timestamp("2024-12-01")]
    forecasted_results = []
    sp_ltv_cac = df[df['transaction_period_ltv_cac'] > 0]['transaction_period_ltv_cac'].min()
    
    for cohort in game_data['cohort'].unique():
        cohort_data = game_data[game_data['cohort'] == cohort].copy()
        m0_ltv_cac = cohort_data['m0_ltv_cac'].iloc[0]
        total_cac = cohort_data['client_actual_spend'].iloc[0]
        last_elapsed_month = int(cohort_data['elapsed_months'].max())
        
        # Add row for elapsed_months=0 with actual data if available
        elapsed_0_data = cohort_data.loc[cohort_data['elapsed_months'] == 0]
        if not elapsed_0_data.empty:
            transactional_actual_0 = elapsed_0_data['transaction_period_ltv_cac'].values[0]
            cumulative_actual_0 = m0_ltv_cac  # Initial cumulative value
            absolute_revenue_actual_0 = total_cac * transactional_actual_0
        else:
            # Default to 0 if no data exists
            transactional_actual_0 = 0
            cumulative_actual_0 = 0
            absolute_revenue_actual_0 = 0
        
        row_result_0 = {
            'game_id': game_id,
            'cohort': cohort,
            'elapsed_months': 0,
            'current_month': cohort.strftime('%Y-%m'),
            'transactional_actual': transactional_actual_0,
            'cumulative_actual': cumulative_actual_0,
            'absolute_revenue_actual': absolute_revenue_actual_0,
            'total_cac': total_cac,
            'transactional_bad': 0,
            'cumulative_bad': cumulative_actual_0,
            'absolute_revenue_bad': absolute_revenue_actual_0,
            'transactional_expected': 0,
            'cumulative_expected': cumulative_actual_0,
            'absolute_revenue_expected': absolute_revenue_actual_0,
            'transactional_good': 0,
            'cumulative_good': cumulative_actual_0,
            'absolute_revenue_good': absolute_revenue_actual_0,
        }
        forecasted_results.append(row_result_0)
        
        # Initialize cumulative values
        cumulative_actual = cumulative_actual_0
        actual_cumulative = []
        for month in range(1, last_elapsed_month + 1):
            historical_data = cohort_data.loc[cohort_data['elapsed_months'] == month, 'transaction_period_ltv_cac']
            transactional_actual = historical_data.values[0] if not historical_data.empty else 0
            cumulative_actual += transactional_actual
            actual_cumulative.append({
                'month': month,
                'transactional_actual': transactional_actual,
                'cumulative_actual': cumulative_actual,
                'absolute_revenue_actual': total_cac * transactional_actual,
                'total_cac': total_cac
            })
        
        # Forecasting logic for elapsed_months >= 1
        lag_1 = np.nan; lag_2 = np.nan; lag_3 = np.nan
        lag_1b = np.nan; lag_1g = np.nan; lag_2b = np.nan; lag_2g = np.nan; lag_3b = np.nan; lag_3g = np.nan
        cumulative_values = {key: cumulative_actual_0 for key in ['cumulative_bad', 'cumulative_expected', 'cumulative_good']}
        
        static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month, best_model_expected.__dict__["feature_names_in_"]].iloc[0].copy()
        for col in ['elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']:
            if col in static_features:
                static_features = static_features.drop(col)
        
        for i in range(1, last_elapsed_month + forecast_months + 1):
            elapsed_months = i
            features = static_features.copy()
            features['elapsed_months'] = elapsed_months
            if 'current_month' in best_model_expected.__dict__["feature_names_in_"]:
                cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
                current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
                features['current_month'] = current_date.month
            else:
                current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)
            
            if i <= last_elapsed_month:
                historical_data = cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac']
                if not historical_data.empty:
                    lag_value = historical_data.values[0] if historical_data.values[0] != 0 else epsilon
                else:
                    lag_value = epsilon
                lag_1 = lag_value
                lag_1b = lag_value
                lag_1g = lag_value
                predicted_val_bad = lag_value
                predicted_val_expected = lag_value
                predicted_val_good = lag_value
            else:
                features1 = features.copy()
                features2 = features.copy()
                features['ltv_lag_1'] = lag_1
                features1['ltv_lag_1'] = lag_1b
                features2['ltv_lag_1'] = lag_1g
                features['ltv_lag_2'] = lag_2
                features1['ltv_lag_2'] = lag_2b
                features2['ltv_lag_2'] = lag_2g
                features['ltv_lag_3'] = lag_3
                features1['ltv_lag_3'] = lag_3b
                features2['ltv_lag_3'] = lag_3g
                features_df = pd.DataFrame([features])
                features_df1 = pd.DataFrame([features1])
                features_df2 = pd.DataFrame([features2])
                predicted_val_bad = max(0, best_model_bad.predict(features_df1)[0])
                predicted_val_expected = max(0, best_model_expected.predict(features_df)[0])
                predicted_val_good = max(0, best_model_good.predict(features_df2)[0])
                x = [predicted_val_bad, predicted_val_expected, predicted_val_good]
                x.sort()
                predicted_val_bad, predicted_val_expected, predicted_val_good = x
                if predicted_val_good <= 0:
                    predicted_val_good = sp_ltv_cac
                if predicted_val_expected <= 0:
                    predicted_val_expected = predicted_val_good / 2
                if predicted_val_bad <= 0:
                    predicted_val_bad = predicted_val_expected / 2
                lag_1b = predicted_val_bad
                lag_1 = predicted_val_expected
                lag_1g = predicted_val_good
            
            lag_3 = lag_2
            lag_2 = lag_1
            lag_3b = lag_2b
            lag_2b = lag_1b
            lag_3g = lag_2g
            lag_2g = lag_1g
            
            row_result = {
                'game_id': game_id,
                'cohort': cohort,
                'elapsed_months': elapsed_months,
                'current_month': current_date.strftime('%Y-%m'),
            }
            actual_data = next((ac for ac in actual_cumulative if ac['month'] == elapsed_months), None)
            row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
            row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
            row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
            row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac
            
            for model_type, predicted_val in zip(['bad', 'expected', 'good'], [predicted_val_bad, predicted_val_expected, predicted_val_good]):
                cumulative_key = f"cumulative_{model_type}"
                cumulative_values[cumulative_key] += predicted_val
                row_result[f"transactional_{model_type}"] = predicted_val
                row_result[cumulative_key] = cumulative_values[cumulative_key]
                row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val
            
            forecasted_results.append(row_result)
    
    forecast_df = pd.DataFrame(forecasted_results)
    return forecast_df

# --- Main Execution for M2 ---
if __name__ == "__main__":
    config_path = "Latest Data/Segregated/config.json"
    try:
        with open(config_path, "r") as f:
            global_config = json.load(f)
    except Exception as e:
        print("No config found or error reading config.json.")
        global_config = {}
    failed_games = []
    changed_features = {}
    for key, cfg in global_config.items():
        if key.startswith("M2_") and cfg.get("flag", 0)==1:
            game_id = cfg["game_id"]
            selected_features = cfg["selected_features"]
            use_augmentation = cfg["use_augmentation"]
            augmentation_factor = cfg.get("augmentation_factor", 1.0)
            augmentation_pct = cfg.get("augmentation_pct", 0)
            print(f"Processing M2 configuration for {game_id} ...")
            try:
                dft, df_orig = preprocess_m2(game_id)
                missing_feats = [feat for feat in selected_features if feat not in dft.columns]
                if missing_feats:
                    print(f"Warning: For game {game_id}, the following features are missing and will be ignored: {missing_feats}")
                    changed_features[game_id] = missing_feats
                    selected_features = [feat for feat in selected_features if feat in dft.columns]
                X = dft[selected_features + ['ltv_lag_1','ltv_lag_2','ltv_lag_3','elapsed_months']]
                y = dft['transaction_period_ltv_cac']
                X_train = X
                y_train = y
                if use_augmentation and augmentation_factor > 1:
                    X_train, y_train = augment_data(X_train, y_train, augmentation_factor, augmentation_pct)
                alphas = [0.25, 0.5, 0.75]
                models_info = {}
                for alpha in alphas:
                    print(f"Training model with alpha={alpha} for game {game_id} ...")
                    result = model_train_save_pipeline(game_id, alpha, X_train, y_train)
                    models_info[alpha] = result
                best_model_expected = models_info[0.5]["model"]
                best_model_bad = models_info[0.75]["model"]
                best_model_good = models_info[0.25]["model"]
                forecast_df = forecast_cumulative_ltv_cac_variations_m2(game_id, best_model_expected, best_model_bad, best_model_good, df_orig, forecast_months=12)
                output_csv = f"{game_id}_predictions.csv"
                forecast_df.to_csv(output_csv, index=False)
                print(f"Predictions saved to {output_csv}")
            except Exception as e:
                print(f"Error processing game {game_id}: {e}")
                failed_games.append(game_id)
                continue
    if failed_games:
        print("\nThe following games failed during training:")
        for game in failed_games:
            print(game)
    if changed_features:
        print("\nThe following games had missing features (these features were ignored and need to be revisited):")
        for game, features in changed_features.items():
            print(f"Game {game}: Missing features {features}")